# Deterministic YOLO cross-validation

This notebook pools the existing `train`, `val`, and `test` directories of an Ultralytics classification dataset and creates deterministic repeated holdout folds. Each fold uses 90% for training, 5% for validation, and 5% for testing by default.

Splitting can be performed by scene ID (to prevent spatial leakage) or by individual image. The target dataset is replaced for every fold, while its manifest and all training, validation, and test results are retained in the configured runs directory.

In [ ]:
from pathlib import Path

# Paths
DATA_ROOT = Path("/path/to/root")
SOURCE = DATA_ROOT / "yolo_dataset_waste_combined_spatial"
TARGET = DATA_ROOT / "yolo_dataset_waste_cv"
RUNS_DIR = DATA_ROOT / "runs/cross_validation"
MODEL_WEIGHTS = "yolo26x-cls.pt"

FOLDS = 5
SEED = 42
VAL_FRACTION = 0.05
TEST_FRACTION = 0.05
SPLIT_MODE = "scene"  # "scene" or "image"

SCENE_REGEX = r"^(?:\d+_yolo_dataset_waste_(?:africa_)?spatial__)?(?P<scene>.+)_\d+$"

TRAINING_PARAMS = {
    "degrees": 0.1,
    "perspective": 0.1,
    "flipud": 0.25,
    "close_mosaic": 20,
    "multi_scale": 0.75,
    "cache": "ram",
    "workers": 16,
    "batch": 64,
    "epochs": 100,
    "imgsz": 128,
    "patience": 20,
}

VALIDATION_PARAMS = {}

In [ ]:
import csv
import json
import math
import random
import re
import shutil
import statistics
import tempfile
from collections import defaultdict
from dataclasses import dataclass
from typing import Any, Callable, Iterable, Mapping, Sequence

from ultralytics import YOLO

SPLITS = ("train", "val", "test")
IMAGE_SUFFIXES = {
    ".avif", ".bmp", ".dng", ".heic", ".jpeg", ".jpg",
    ".mpo", ".png", ".tif", ".tiff", ".webp",
}

@dataclass(frozen=True)
class ImageRecord:
    path: Path
    source_relative_path: Path
    class_name: str
    unit_id: str

def make_scene_id_parser(pattern: str | None) -> Callable[[Path], str]:
    if pattern is None:
        def parse(path: Path) -> str:
            if "_" not in path.stem:
                raise ValueError(
                    f"Cannot derive a scene ID from {path.name!r}. "
                    "Set SCENE_REGEX or use SPLIT_MODE = 'image'."
                )
            return path.stem.rsplit("_", 1)[0]
        return parse

    regex = re.compile(pattern)
    def parse(path: Path) -> str:
        match = regex.search(path.stem)
        if match is None:
            raise ValueError(f"Filename {path.name!r} does not match {pattern!r}")
        if "scene" in match.groupdict():
            return match.group("scene")
        return match.group(1) if match.lastindex else match.group(0)
    return parse

def discover_images(source: Path, split_mode: str, scene_regex: str | None) -> list[ImageRecord]:
    if split_mode not in {"scene", "image"}:
        raise ValueError("SPLIT_MODE must be 'scene' or 'image'")
    if not source.is_dir():
        raise FileNotFoundError(f"Dataset does not exist: {source}")

    scene_parser = make_scene_id_parser(scene_regex)
    records = []
    destination_names = set()
    for old_split in SPLITS:
        split_dir = source / old_split
        if not split_dir.is_dir():
            raise ValueError(f"Missing source split: {split_dir}")
        for class_dir in sorted(path for path in split_dir.iterdir() if path.is_dir()):
            for image in sorted(class_dir.rglob("*")):
                if not image.is_file() or image.suffix.lower() not in IMAGE_SUFFIXES:
                    continue
                relative = image.relative_to(source)
                destination_key = (class_dir.name, image.name)
                if destination_key in destination_names:
                    raise ValueError(
                        f"Duplicate filename {image.name!r} in class {class_dir.name!r}"
                    )
                destination_names.add(destination_key)
                unit_id = scene_parser(image) if split_mode == "scene" else relative.as_posix()
                records.append(ImageRecord(image, relative, class_dir.name, unit_id))

    if not records:
        raise ValueError(f"No supported images found below {source}")
    return records

def fraction_count(total: int, fraction: float) -> int:
    if not 0 <= fraction < 1:
        raise ValueError("Split fractions must be in [0, 1)")
    return 0 if fraction == 0 else max(1, round(total * fraction))

def assign_units(
    unit_ids: Iterable[str], fold: int, seed: int,
    val_fraction: float, test_fraction: float,
) -> dict[str, str]:
    units = sorted(set(unit_ids))
    random.Random(seed).shuffle(units)
    n_val = fraction_count(len(units), val_fraction)
    n_test = fraction_count(len(units), test_fraction)
    if n_val + n_test >= len(units):
        raise ValueError("Not enough splitting units to leave a training set")

    # Rotate through consecutive blocks of one deterministic random ordering.
    holdout_size = n_val + n_test
    start = (fold * holdout_size) % len(units)
    held_out = [units[(start + index) % len(units)] for index in range(holdout_size)]
    validation = set(held_out[:n_val])
    testing = set(held_out[n_val:])
    return {
        unit: "val" if unit in validation else "test" if unit in testing else "train"
        for unit in units
    }

In [ ]:
def validate_paths(source: Path, target: Path, runs_dir: Path) -> None:
    source, target, runs_dir = source.resolve(), target.resolve(), runs_dir.resolve()
    if source == target or source in target.parents or target in source.parents:
        raise ValueError("SOURCE and TARGET must be separate, non-nested directories")
    if runs_dir in {source, target}:
        raise ValueError("RUNS_DIR must differ from SOURCE and TARGET")
    if runs_dir in target.parents or target in runs_dir.parents:
        raise ValueError("RUNS_DIR and TARGET must not contain one another")
    if target == Path(target.anchor):
        raise ValueError("Refusing to overwrite a filesystem root")

def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)

def materialize_fold(
    source: Path, target: Path, records: Sequence[ImageRecord],
    assignments: Mapping[str, str], fold: int, seed: int, split_mode: str,
) -> dict[str, Any]:
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(tempfile.mkdtemp(prefix=f".{target.name}-", dir=target.parent))
    class_names = sorted({record.class_name for record in records})
    image_counts = {split: 0 for split in SPLITS}
    unit_counts = {split: 0 for split in SPLITS}
    rows = []

    try:
        for split in SPLITS:
            for class_name in class_names:
                (temporary / split / class_name).mkdir(parents=True, exist_ok=True)

        for record in records:
            split = assignments[record.unit_id]
            destination = temporary / split / record.class_name / record.path.name
            shutil.copy2(record.path, destination)
            image_counts[split] += 1
            rows.append([
                fold, seed, split_mode, record.unit_id, split, record.class_name,
                record.source_relative_path.as_posix(),
                destination.relative_to(temporary).as_posix(),
            ])

        for split in assignments.values():
            unit_counts[split] += 1

        with (temporary / "split_manifest.csv").open("w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow([
                "fold", "seed", "split_mode", "unit_id", "split",
                "class_name", "source", "destination",
            ])
            writer.writerows(sorted(rows, key=lambda row: (row[4], row[5], row[6])))

        summary = {
            "fold": fold, "seed": seed, "split_mode": split_mode,
            "image_counts": image_counts, "unit_counts": unit_counts,
        }
        (temporary / "split_summary.json").write_text(
            json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"
        )
        remove_path(target)
        temporary.replace(target)
        return summary
    except BaseException:
        remove_path(temporary)
        raise

def metric_dict(result: Any) -> dict[str, Any]:
    raw = result if isinstance(result, Mapping) else getattr(result, "results_dict", {})
    metrics = {}
    for key, value in raw.items():
        if hasattr(value, "item"):
            try:
                value = value.item()
            except (TypeError, ValueError, RuntimeError):
                continue
        if value is None or isinstance(value, (bool, int, float, str)):
            metrics[str(key)] = value
    return metrics

def aggregate_metrics(fold_results: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    values = defaultdict(list)
    for fold_result in fold_results:
        for phase in ("train_metrics", "val_metrics", "test_metrics"):
            for name, value in fold_result.get(phase, {}).items():
                if (isinstance(value, (int, float)) and not isinstance(value, bool)
                        and math.isfinite(value)):
                    values[f"{phase}.{name}"].append(float(value))
    return {
        name: {
            "mean": statistics.fmean(samples),
            "std": statistics.pstdev(samples),
            "min": min(samples),
            "max": max(samples),
            "count": len(samples),
        }
        for name, samples in sorted(values.items())
    }

def write_metric_outputs(runs_dir: Path, fold_results: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    aggregate = aggregate_metrics(fold_results)
    payload = {"folds": list(fold_results), "aggregate": aggregate}
    (runs_dir / "metrics.json").write_text(
        json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )

    metric_names = sorted(aggregate)
    with (runs_dir / "metrics.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["fold", *metric_names])
        writer.writeheader()
        for fold_result in fold_results:
            row = {"fold": fold_result["fold"]}
            for phase in ("train_metrics", "val_metrics", "test_metrics"):
                for name, value in fold_result.get(phase, {}).items():
                    row[f"{phase}.{name}"] = value
            writer.writerow(row)
        writer.writerow({"fold": "mean", **{k: v["mean"] for k, v in aggregate.items()}})
        writer.writerow({"fold": "std", **{k: v["std"] for k, v in aggregate.items()}})
    return aggregate

In [ ]:
def cross_validate(
    source: Path, target: Path, runs_dir: Path, model_weights: str | Path,
    training_params: Mapping[str, Any], *, folds: int = 5, seed: int = 42,
    val_fraction: float = 0.05, test_fraction: float = 0.05,
    split_mode: str = "scene", scene_regex: str | None = None,
    validation_params: Mapping[str, Any] | None = None,
) -> dict[str, Any]:
    if folds < 1:
        raise ValueError("FOLDS must be at least 1")
    validate_paths(source, target, runs_dir)
    records = discover_images(source, split_mode, scene_regex)
    unit_ids = {record.unit_id for record in records}
    runs_dir.mkdir(parents=True, exist_ok=True)

    train_args = dict(training_params)
    train_args.setdefault("seed", seed)
    val_args = dict(validation_params or {})
    val_args.setdefault("imgsz", train_args.get("imgsz", 128))
    val_args.setdefault("batch", train_args.get("batch", 64))
    val_args.setdefault("workers", train_args.get("workers", 16))

    config = {
        "source": str(source.resolve()), "target": str(target.resolve()),
        "runs_dir": str(runs_dir.resolve()), "model_weights": str(model_weights),
        "folds": folds, "seed": seed, "val_fraction": val_fraction,
        "test_fraction": test_fraction, "split_mode": split_mode,
        "scene_regex": scene_regex, "training_params": train_args,
        "validation_params": val_args,
    }
    (runs_dir / "run_config.json").write_text(
        json.dumps(config, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )

    fold_results = []
    for fold_index in range(folds):
        fold_number = fold_index + 1
        fold_dir = runs_dir / f"fold_{fold_number:02d}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        assignments = assign_units(
            unit_ids, fold_index, seed, val_fraction, test_fraction
        )
        split_summary = materialize_fold(
            source, target, records, assignments, fold_number, seed, split_mode
        )
        shutil.copy2(target / "split_manifest.csv", fold_dir / "split_manifest.csv")
        shutil.copy2(target / "split_summary.json", fold_dir / "split_summary.json")

        print(f"\n{'=' * 20} Fold {fold_number}/{folds} {'=' * 20}")
        print(split_summary)

        model = YOLO(str(model_weights))
        train_metrics = model.train(
            data=str(target), project=str(fold_dir), name="train",
            exist_ok=True, **train_args,
        )
        best_weights = Path(model.trainer.best)
        best_model = YOLO(str(best_weights))
        val_metrics = metric_dict(best_model.val(
            data=str(target), split="val", project=str(fold_dir),
            name="validation", exist_ok=True, **val_args,
        ))
        test_metrics = metric_dict(best_model.val(
            data=str(target), split="test", project=str(fold_dir),
            name="test", exist_ok=True, **val_args,
        ))

        fold_result = {
            "fold": fold_number, "split": split_summary,
            "best_weights": str(best_weights),
            "train_metrics": metric_dict(train_metrics),
            "val_metrics": val_metrics,
            "test_metrics": test_metrics,
        }
        fold_results.append(fold_result)
        (fold_dir / "metrics.json").write_text(
            json.dumps(fold_result, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )
        write_metric_outputs(runs_dir, fold_results)

    return {
        "config": config, "folds": fold_results,
        "aggregate": aggregate_metrics(fold_results),
    }

## Run all folds


In [ ]:
results = cross_validate(
    source=SOURCE,
    target=TARGET,
    runs_dir=RUNS_DIR,
    model_weights=MODEL_WEIGHTS,
    training_params=TRAINING_PARAMS,
    folds=FOLDS,
    seed=SEED,
    val_fraction=VAL_FRACTION,
    test_fraction=TEST_FRACTION,
    split_mode=SPLIT_MODE,
    scene_regex=SCENE_REGEX,
    validation_params=VALIDATION_PARAMS,
)

In [ ]:
results["aggregate"]